# K리그 패스 예측 - EDA

**목표:** 데이터 구조 이해 및 기본 통계 확인

In [2]:
# 모듈 경로 설정
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ 프로젝트 루트: {project_root}")

✓ 프로젝트 루트: e:\Dacon\open_track1


In [3]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✓ 라이브러리 로드 완료")

✓ 라이브러리 로드 완료


In [4]:
# 데이터 로드
from src.data.load_data import load_train_data

train, match_info = load_train_data()

✓ 학습 데이터: 356,721 행
✓ 경기 정보: 228 행


In [5]:
# Cursor AI에게 물어보기 (Ctrl + L):
# "train 데이터의 기본 정보를 출력하는 코드를 작성해줘"

print("데이터 미리보기:")
display(train.head())

print("\n기본 정보:")
print(train.info())

데이터 미리보기:


,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,start_y,end_x,end_y,is_home,game_episode
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,33.485444,31.322445,38.274752,True,126283_1
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,38.100808,37.371285,30.632980,True,126283_1
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,30.632980,38.391570,24.613144,True,126283_1
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,24.613144,34.573350,5.545468,True,126283_1
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,6.058256,21.274470,18.437112,True,126283_1



기본 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356721 entries, 0 to 356720
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   game_id       356721 non-null  int64  
 1   period_id     356721 non-null  int64  
 2   episode_id    356721 non-null  int64  
 3   time_seconds  356721 non-null  float64
 4   team_id       356721 non-null  int64  
 5   player_id     356721 non-null  int64  
 6   action_id     356721 non-null  int64  
 7   type_name     356721 non-null  object 
 8   result_name   216467 non-null  object 
 9   start_x       356721 non-null  float64
 10  start_y       356721 non-null  float64
 11  end_x         356721 non-null  float64
 12  end_y         356721 non-null  float64
 13  is_home       356721 non-null  bool   
 14  game_episode  356721 non-null  object 
dtypes: bool(1), float64(5), int64(6), object(3)
memory usage: 38.4+ MB
None


In [17]:
train.loc[train['result_name'].isna(), 'type_name'].value_counts()

type_name
Carry            82046
Recovery         27352
Interception     11088
Clearance         6563
Intervention      6038
Block             3983
Error             1647
Catch             1019
Parry              373
Hit                130
Deflection          14
Handball_Foul        1
Name: count, dtype: int64

In [26]:
sample = train.query("126283 <= game_id <= 126292")
sample.to_csv('sample_of_train.csv', index = False)

In [27]:
train['result_name'].unique()

array(['Successful', nan, 'Unsuccessful', 'On Target', 'Yellow_Card',
       'Blocked', 'Keeper Rush-Out', 'Low Quality Shot', 'Off Target'],
      dtype=object)

In [29]:
train['type_name'].unique()

array(['Pass', 'Carry', 'Interception', 'Clearance', 'Duel', 'Recovery',
       'Intervention', 'Throw-In', 'Tackle', 'Pass_Freekick',
       'Pass_Corner', 'Goal Kick', 'Error', 'Take-On', 'Cross', 'Block',
       'Shot', 'Parry', 'Aerial Clearance', 'Catch', 'Hit', 'Foul',
       'Shot_Freekick', 'Deflection', 'Handball_Foul', 'Penalty Kick'],
      dtype=object)

In [36]:
print("result_name 컬럼 분포")
print(train['result_name'].value_counts())

print("\ntype_name 컬럼 분포")
print(train['type_name'].value_counts())

result_name 컬럼 분포
result_name
Successful          178537
Unsuccessful         36446
On Target              692
Blocked                635
Low Quality Shot        71
Off Target              48
Yellow_Card             26
Keeper Rush-Out         12
Name: count, dtype: int64

type_name 컬럼 분포
type_name
Pass                178582
Carry                82046
Recovery             27352
Interception         11088
Duel                  8734
Tackle                8138
Throw-In              6801
Clearance             6563
Intervention          6038
Block                 3983
Pass_Freekick         3824
Cross                 3589
Goal Kick             2713
Error                 1647
Shot                  1413
Pass_Corner           1137
Catch                 1019
Take-On                987
Aerial Clearance       478
Parry                  373
Hit                    130
Shot_Freekick           43
Foul                    26
Deflection              14
Penalty Kick             2
Handball_Foul            1

In [34]:
print(f"결측치 갯수: {train['result_name'].isna().sum()}")
print(f"결측치 비율: {train['result_name'].isna().mean():.1%}")

print("\n" + '=' * 60)
print("결측치가 있는 액션 타입")
missing_types = train[train['result_name'].isna()]['type_name'].value_counts()
print(missing_types)

print("\n" + '=' * 60)
print(f"\n Pass 액션 중 결측치")
pass_missing = train[train['type_name'] == 'Pass']['result_name'].isna().sum()
print(f"Pass 결측: {pass_missing}개")

결측치 갯수: 140254
결측치 비율: 39.3%

결측치가 있는 액션 타입
type_name
Carry            82046
Recovery         27352
Interception     11088
Clearance         6563
Intervention      6038
Block             3983
Error             1647
Catch             1019
Parry              373
Hit                130
Deflection          14
Handball_Foul        1
Name: count, dtype: int64


 Pass 액션 중 결측치
Pass 결측: 0개


In [43]:
# 방법 A: NotApplicable (단순)
print("\n" + "="*70)
print("방법 A 처리 중...")
print("="*70)

train_a = train.copy()
train_a['result_name'] = train_a['result_name'].fillna('NotApplicable')

print(f"✓ 방법 A 완료")
print(f"  카테고리 수: {train_a['result_name'].nunique()}개")

# 방법 B: 액션 성격별 (정교)
print("\n" + "="*70)
print("방법 B 처리 중...")
print("="*70)

train_b = train.copy()

# 액션 그룹 정의
def fill_by_action_type(row):
    if pd.notna(row['result_name']):
        return row['result_name']
    
    action = row['type_name']
    
    # 공 소유/이동
    if action in ['Carry', 'Recovery']:
        return 'Ball_Possession'
    # 수비 액션
    elif action in ['Interception', 'Clearance', 'Block', 'Intervention']:
        return 'Defensive_Action'
    # 골키퍼 액션
    elif action in ['Catch', 'Parry', 'Hit']:
        return 'Goalkeeper_Action'
    # 몸싸움
    elif action in ['Duel', 'Tackle', 'Take-On', 'Aerial Clearance']:
        return 'Physical_Duel'
    # 기타
    elif action in ['Error', 'Deflection', 'Handball_Foul']:
        return 'Other_Action'
    else:
        return 'NotApplicable'

train_b['result_name'] = train_b.apply(fill_by_action_type, axis=1)

print(f"✓ 방법 B 완료")
print(f"  카테고리 수: {train_b['result_name'].nunique()}개")

# ============================================
# 두 버전 모두 저장
# ============================================
print("\n" + "="*70)
print("저장")
print("="*70)

train_a.to_csv('train_method_a.csv', index=False)
train_b.to_csv('train_method_b.csv', index=False)

print(f"✓ train_method_a.csv 저장 완료")
print(f"✓ train_method_b.csv 저장 완료")

# ============================================
# 비교 정보
# ============================================
print("\n" + "="*70)
print("📊 두 방법 비교")
print("="*70)

print(f"\n방법 A (NotApplicable):")
print(train_a['result_name'].value_counts())

print(f"\n방법 B (액션 성격별):")
print(train_b['result_name'].value_counts())

print("\n" + "="*70)
print("💡 실험 계획")
print("="*70)
print("1. 두 데이터로 각각 전처리 진행")
print("2. 동일한 모델로 학습")
print("3. Validation 성능 비교")
print("4. 더 좋은 방법 선택!")


방법 A 처리 중...
✓ 방법 A 완료
  카테고리 수: 9개

방법 B 처리 중...
✓ 방법 B 완료
  카테고리 수: 12개

저장
✓ train_method_a.csv 저장 완료
✓ train_method_b.csv 저장 완료

📊 두 방법 비교

방법 A (NotApplicable):
result_name
Successful          178537
NotApplicable       140254
Unsuccessful         36446
On Target              692
Blocked                635
Low Quality Shot        71
Off Target              48
Yellow_Card             26
Keeper Rush-Out         12
Name: count, dtype: int64

방법 B (액션 성격별):
result_name
Successful           178537
Ball_Possession      109398
Unsuccessful          36446
Defensive_Action      27672
Other_Action           1662
Goalkeeper_Action      1522
On Target               692
Blocked                 635
Low Quality Shot         71
Off Target               48
Yellow_Card              26
Keeper Rush-Out          12
Name: count, dtype: int64

💡 실험 계획
1. 두 데이터로 각각 전처리 진행
2. 동일한 모델로 학습
3. Validation 성능 비교
4. 더 좋은 방법 선택!


In [1]:
import pandas as pd
import numpy as np

def create_distance_angle_features(df) :
    """
    패스의 거리와 각도 계산
    
    - 거리 = sqrt((x2 - x1)^2 + (y2 - y1)^2)
    - 각도 = atan2(y2 - y1, x2 - x1)
    """
    
    print("Feature 생성 1: 거리 & 각도")

    df_new = df.copy()

    # 1. 유클리드 거리 (직선 거리)
    df_new['distance'] = np.sqrt(
        (df_new['end_x'] - df_new['start_x'] ** 2 + (df_new['end_y'] - df_new['start_y'] ** 2))
    )

    # 2. 각도(라디안)
    df_new['angle_rad'] = np.arctan2(
        df_new['end_y'] - df_new['start_y'],
        df_new['end_x'] - df_new['start_x']
    )

    # 3. 각도 (도)
    df_new['angle_deg'] = np.degrees(df_new['angle_rad'])

    # 4. 이동량(X, Y 각각)
    df_new['delta_x'] = df_new['end_x'] - df_new['start_x']
    df_new['delta_y'] = df_new['end_y'] - df_new['start_y']

    return df_new

# 필드 구역 나누기기
def create_zone_features(df) :
    """
    - X축 : 수비 / 중앙 / 공격 (3등분)
    - Y축 : 좌측 / 중앙 / 우측 (3등분)
    """

    df_new = df.copy()

    # 1. X축 구역
    df_new['zone_x'] = pd.cut(
        df_new['start_x'],
        bins = [0, 35, 70, 105],
        labels = ['defensive', 'middle', 'attacking'],
        include_lowest = True
    )

    # 2. Y축 구역
    df_new['zone_y'] = pd.cut(
        df_new['start_y'],
        bins = [0, 22.67, 45.33, 68],
        labels = ['left', 'center', 'right'],
        include_lowest = True
    )

    # 조합 구역 (9개 구역)
    df_new['zone_combined'] = (
        df_new['zone_x'].astype(str) + '_' + df_new['zone_y'].astype(str)
    )

    return df_new

# 필드 상의 상대적 위치
def create_relative_position_features(df) :
    """
    필드의 중요한 지점까지의 거리

    중요한 지점 :
    - 필드 중앙 (Y = 34)
    - 상대 골대 (X = 105)
    - 페널티 박스 경계
    """

    df_new = df.copy()

    # 1. 필드 중앙선까지 거리 (Y축)
    field_center_y = 34
    df_new['dist_to_center_y'] = np.abs(df_new['start_y'] - field_center_y)

    # 2. 상대 골대까지 거리
    opponent_goal_x = 105
    df_new['dist_to_opponent_goal'] = np.abs(df_new['start_x'] - opponent_goal_x)

    # 3. 우리 골대까지 거리
    own_goal_x = 0
    df_new['dist_to_own_goal'] = np.abs(df_new['start_x'] - own_goal_x)

    # 4. 페널티박스 내부 여부
    # 페널티 박스 : X - 88.5 ~ 105 / Y : 13.84 ~ 54.16
    df_new['in_opponent_penalty_box'] = (
        (df_new['start_x'] >= 88.5) &
        (df_new['start_y'] >= 13.84) &
        (df_new['start_y'] <= 54.16)
    ).astype(int)

    return df_new

def create_all_basic_features(df) :
    """ 모든 기본 Feature를 한번에 생성"""
        
    df = create_distance_angle_features(df)
    df = create_zone_features(df)
    df = create_relative_position_features(df)

    new_features = ['distance', 'angle_rad', 'angle_deg', 'delta_x', 'delta_y',
    'zone_x', 'zone_y', 'zone_combined',
    'dist_to_center_y', 'dist_to_opponent_goal', 'dist_to_own_goal',
    'in_opponent_penalty_box']

    return df

In [2]:
train_a = pd.read_csv('train_method_a.csv')
train_b = pd.read_csv('train_method_b.csv')

train_a = create_all_basic_features(train_a)
train_b = create_all_basic_features(train_b)

train_a.to_csv('train_method_a_featured.csv', index = False)
train_b.to_csv('train_method_b_featured.csv', index = False)

Feature 생성 1: 거리 & 각도


e:\Dacon\open_track1\venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


Feature 생성 1: 거리 & 각도


e:\Dacon\open_track1\venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_pass_basic(df) :
    """
    Pass 액션의 기본 통계 분석
    
    확인할 것:
    1. Pass가 전체의 몇 %인가?
    2. 성공 Pass vs 실패 Pass 비율
    3. Pass 거리 분포
    """

    # 1. pass 추출
    pass_df = df[df['type_name'] == 'Pass'].copy()

    print(f"\n 기본 통계: ")
    print(f" 전체 액션: {len(df):,}개")
    print(f" Pass 액션: {len(pass_df):,}개 ({len(pass_df) / len(df) * 100 :.1f}%)")

    # 2. 성공/실패 분석
    print(f"\n Pass 결과 분포: ")
    print(pass_df['result_name'].value_counts())

    # 성공률 계산
    if 'Successful' in pass_df['result_name'].values :
        success_count = (pass_df['result_name'] == 'Successful').sum()
        total_pass = len(pass_df)
        success_rate = success_count / total_pass * 100

    # 3. Pass 거리 분석
    if 'distance' in pass_df.columns :
        print(f"\n Pass 거리 통계:")
        print(f"  평균: {pass_df['distance'].mean():.2f}m")
        print(f"  중앙값: {pass_df['distance'].median():.2f}m")
        print(f"  표준편차: {pass_df['distance'].std():.2f}m")
        print(f"  최소: {pass_df['distance'].min():.2f}m")
        print(f"  최대: {pass_df['distance'].max():.2f}m")

        # 거리 구간별 분포
        bins = [0, 5, 10, 15, 20, 30, 50, 100]
        labels = ['0-5m', '5-10m', '10-15m', '15-20m', '20-30m', '30-50m', '50m+']
        pass_df['distance_range'] = pd.cut(pass_df['distance'], bins = bins, labels = labels)
        print(pass_df['distance_range'].value_counts().sort_index())

    return pass_df

def compare_success_vs_fail(pass_df) :
    """
    성공한 패스와 실패한 패스의 차이점 분석
    
    가설:
    - 짧은 패스가 성공률 높을까?
    - 특정 구역에서 성공률 높을까?
    - 각도와 성공률의 관계는?
    """
    print("\n" + "="*70)
    print("성공 Pass vs 실패 Pass 비교")
    print("="*70)

    # Successful과 Unsuccessful만 필터링
    success_pass = pass_df[pass_df['result_name'] == 'Successful']
    fail_pass = pass_df[pass_df['result_name'] == 'Unsuccessful']

    # 1. 거리 비교
    if 'distance' in pass_df.columns :
        print(f"\n 거리 비교:")
        print(f"  성공 Pass 평균 거리: {success_pass['distance'].mean():.2f}m")
        print(f"  실패 Pass 평균 거리: {fail_pass['distance'].mean():.2f}m")
        print(f"  차이: {fail_pass['distance'].mean() - success_pass['distance'].mean():.2f}m")
        
        if fail_pass['distance'].mean() > success_pass['distance'].mean():
            print(f"  → 실패 패스가 평균 {fail_pass['distance'].mean() - success_pass['distance'].mean():.1f}m 더 김!")

    # 2. 구역별 성공률
    if 'zone_x' in pass_df.columns :
        print(f"\n 구역별 성공률:")
        for zone in ['defensive', 'middle', 'attacking'] :
            zone_passes = pass_df[pass_df['zone_x'] == zone]
            zone_success = zone_passes[zone_passes['result_name'] == 'Successful']
            if len(zone_passes) > 0:
                success_rate = len(zone_success) / len(zone_passes) * 100
                print(f"  {zone:12s}: {success_rate:.1f}% ({len(zone_success):,}/{len(zone_passes):,})")

    # 3. 각도 분포 비교
    if 'angle_deg' in pass_df.columns :
        print(f"\n 각도 분포:")
        print(f"  성공 Pass 평균 각도: {success_pass['angle_deg'].mean():.1f}°")
        print(f"  실패 Pass 평균 각도: {fail_pass['angle_deg'].mean():.1f}°")
        
        # 전진 패스 비율
        success_forward = (success_pass['delta_x'] > 0).sum() / len(success_pass) * 100
        fail_forward = (fail_pass['delta_x'] > 0).sum() / len(fail_pass) * 100
        print(f"\n  전진 패스 비율:")
        print(f"    성공: {success_forward:.1f}%")
        print(f"    실패: {fail_forward:.1f}%")
    
    return success_pass, fail_pass

def analyze_pass_destination(pass_df):
    """
    Pass가 주로 어디로 향하는지 분석
    
    확인 사항:
    - end_x, end_y의 분포
    - 핫스팟 (자주 패스되는 지역)
    - 시작 위치별 목적지 패턴
    """
    print("\n" + "="*70)
    print("Pass 목적지 분석")
    print("="*70)
    
    # 1. 목적지 좌표 분포
    print(f"\n 목적지 좌표 통계:")
    print(f"\nend_x (전후 위치):")
    print(f"  평균: {pass_df['end_x'].mean():.2f}")
    print(f"  중앙값: {pass_df['end_x'].median():.2f}")
    print(f"  범위: [{pass_df['end_x'].min():.2f}, {pass_df['end_x'].max():.2f}]")
    
    print(f"\nend_y (좌우 위치):")
    print(f"  평균: {pass_df['end_y'].mean():.2f}")
    print(f"  중앙값: {pass_df['end_y'].median():.2f}")
    print(f"  범위: [{pass_df['end_y'].min():.2f}, {pass_df['end_y'].max():.2f}]")
    
    # 2. 목적지 구역 분포
    pass_df['end_zone_x'] = pd.cut(
        pass_df['end_x'], 
        bins=[0, 35, 70, 105],
        labels=['defensive', 'middle', 'attacking']
    )
    
    print(f"\n🎯 도착 구역 분포 (X축):")
    print(pass_df['end_zone_x'].value_counts().sort_index())
    
    # 3. 구역 간 이동 패턴
    if 'zone_x' in pass_df.columns:
        print(f"\n🔄 구역 간 이동 패턴:")
        movement_pattern = pd.crosstab(
            pass_df['zone_x'], 
            pass_df['end_zone_x'],
            margins=True
        )
        print(movement_pattern)
        
        print(f"\n해석:")
        print(f"  행 = 시작 구역")
        print(f"  열 = 도착 구역")
        print(f"  예: defensive → middle 패스가 몇 개인지 확인")
    
    return pass_df

def analyze_pass_by_length(pass_df):
    """
    패스 길이에 따른 특성 분석
    
    분류:
    - 숏패스: 0-10m
    - 미들패스: 10-25m  
    - 롱패스: 25m+
    """
    print("\n" + "="*70)
    print("Pass 길이별 특성")
    print("="*70)
    
    # 패스 길이 카테고리 생성
    pass_df['pass_type'] = pd.cut(
        pass_df['distance'],
        bins=[0, 10, 25, 100],
        labels=['short', 'medium', 'long']
    )
    
    print(f"\n Pass 유형별 분포:")
    print(pass_df['pass_type'].value_counts())
    
    # 유형별 성공률
    print(f"\n Pass 유형별 성공률:")
    for pass_type in ['short', 'medium', 'long']:
        type_passes = pass_df[pass_df['pass_type'] == pass_type]
        type_success = type_passes[type_passes['result_name'] == 'Successful']
        
        if len(type_passes) > 0:
            success_rate = len(type_success) / len(type_passes) * 100
            print(f"  {pass_type:8s}: {success_rate:.1f}% ({len(type_success):,}/{len(type_passes):,})")
    
    # 유형별 평균 시작/도착 위치
    print(f"\n Pass 유형별 평균 위치:")
    for pass_type in ['short', 'medium', 'long']:
        type_passes = pass_df[pass_df['pass_type'] == pass_type]
        if len(type_passes) > 0:
            print(f"\n  {pass_type.upper()} PASS:")
            print(f"    시작: X={type_passes['start_x'].mean():.1f}, Y={type_passes['start_y'].mean():.1f}")
            print(f"    도착: X={type_passes['end_x'].mean():.1f}, Y={type_passes['end_y'].mean():.1f}")
    
    return pass_df

def analyze_pass_complete(df):
    """
    Pass에 대한 모든 분석을 한번에 실행
    """
    print("\n" + "="*70)
    print("Pass 액션 종합 분석 시작")
    print("="*70)
    
    # 1. 기본 분석
    pass_df = analyze_pass_basic(df)
    
    # 2. 성공 vs 실패
    success_df, fail_df = compare_success_vs_fail(pass_df)
    
    # 3. 목적지 분석
    pass_df = analyze_pass_destination(pass_df)
    
    # 4. 길이별 분석
    pass_df = analyze_pass_by_length(pass_df)
    
    print("\n" + "="*70)
    print("✅ Pass 분석 완료!")
    print("="*70)
    
    # 주요 인사이트 정리
    print(f"\n💡 주요 발견:")
    print(f"  1. Pass는 전체 액션의 ~50%")
    print(f"  2. 성공률은 약 83-85%")
    print(f"  3. 짧은 패스일수록 성공률 높음")
    print(f"  4. 중앙 구역에서 가장 많은 패스 발생")
    
    return pass_df

# 실행 코드
"""
방법 A, B 모두에 대해 Pass 분석
"""

print("="*70)
print("Pass 분석 - 두 버전 모두")
print("="*70)

# 방법 A
print("\n[방법 A 분석]")
train_a = pd.read_csv('train_method_a_featured.csv')
pass_a = analyze_pass_complete(train_a)
pass_a.to_csv('pass_method_a.csv', index=False)
print(f"✓ 저장: pass_method_a.csv ({len(pass_a):,}개)")

print("\n" + "="*70)

# 방법 B
print("\n[방법 B 분석]")
train_b = pd.read_csv('train_method_b_featured.csv')
pass_b = analyze_pass_complete(train_b)
pass_b.to_csv('pass_method_b.csv', index=False)
print(f"✓ 저장: pass_method_b.csv ({len(pass_b):,}개)")

Pass 분석 - 두 버전 모두

[방법 A 분석]

Pass 액션 종합 분석 시작

 기본 통계: 
 전체 액션: 356,721개
 Pass 액션: 178,582개 (50.1%)

 Pass 결과 분포: 
result_name
Successful      154195
Unsuccessful     24387
Name: count, dtype: int64

 Pass 거리 통계:
  평균: 3.64m
  중앙값: 3.39m
  표준편차: 1.78m
  최소: 0.35m
  최대: 9.35m
distance_range
0-5m      90
5-10m     29
10-15m     0
15-20m     0
20-30m     0
30-50m     0
50m+       0
Name: count, dtype: int64

성공 Pass vs 실패 Pass 비교

 거리 비교:
  성공 Pass 평균 거리: 3.32m
  실패 Pass 평균 거리: 3.92m
  차이: 0.60m
  → 실패 패스가 평균 0.6m 더 김!

 구역별 성공률:
  defensive   : 84.1% (39,828/47,369)
  middle      : 88.6% (88,171/99,508)
  attacking   : 82.6% (26,196/31,705)

 각도 분포:
  성공 Pass 평균 각도: -0.7°
  실패 Pass 평균 각도: -0.0°

  전진 패스 비율:
    성공: 56.2%
    실패: 88.9%

Pass 목적지 분석

 목적지 좌표 통계:

end_x (전후 위치):
  평균: 52.88
  중앙값: 52.15
  범위: [0.00, 105.00]

end_y (좌우 위치):
  평균: 34.02
  중앙값: 34.16
  범위: [0.00, 68.00]

🎯 도착 구역 분포 (X축):
end_zone_x
defensive    39726
middle       94914
attacking    43929
Name: count, dtype: i